## Rect Grad Optimization

In [17]:
%matplotlib qt
import torch
import coilsim
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

In [9]:
# Gradient boards
separation = 0.03
width = 0.04
length = 0.2
# length = 0.05
turns = 10
spacing = 0.001 
board_thickness = 0.0016

# heights = torch.arange(0,-length/2,-length/2/(turns), requires_grad=True)
raw_heights = torch.ones(turns, requires_grad=True)

points_combined = torch.zeros(((turns*4+1)*2-4, 3), requires_grad=True)
points_sc = torch.zeros(((turns*4+1)*2-2, 3), requires_grad=True)

def create_gradcoil(raw_heights, spacing, points1, points2):

    pos_heights = torch.nn.Softplus()(raw_heights)
    heights = (-(torch.cumsum(pos_heights, dim=0))/torch.sum(pos_heights + spacing*turns) * 
                 (length/2 - length/2/(turns) - spacing*turns))
    heights = heights - torch.arange(1, len(heights) + 1) * spacing

    # Build all points as a function of heights
    point_list = []
    for i in range(turns-1, -1, -1):

        s = spacing * i
        h = heights[i]
        
        point_list.append(torch.stack([
            torch.tensor(width/2 - s - spacing/2),
            h,
            torch.tensor(separation/2 + board_thickness/2)
        ]))
        point_list.append(torch.stack([
            torch.tensor(-width/2 + s - spacing/2),
            h,
            torch.tensor(separation/2 + board_thickness/2)
        ]))
        point_list.append(torch.stack([
            torch.tensor(-width/2 + s - spacing/2),
            torch.tensor(-length/2 + s - spacing),
            torch.tensor(separation/2 + board_thickness/2)
        ]))
        point_list.append(torch.stack([
            torch.tensor(width/2 - s + spacing - spacing/2),
            torch.tensor(-length/2 + s - spacing),
            torch.tensor(separation/2 + board_thickness/2)
        ]))

    point_list.append(torch.tensor([width/2 - s + spacing - spacing/2,
                                    0,
                                    separation/2 + board_thickness/2], dtype=torch.float32))


    points = torch.stack(point_list)

    # Flip and negate x,y coordinates (no in-place operations)
    points_flipped = torch.flip(points, dims=[0])
    points_flipped = torch.cat([-points_flipped[:, :2], points_flipped[:, 2:]], dim=1)

    # Concatenate (no in-place operations)
    points_top = torch.cat([points, points_flipped], dim=0)

    points_bottom = torch.flip(points_top, dims=[0])
    points_bottom = torch.cat([points_bottom[1:-1,0:1], -points_bottom[1:-1,1:2], points_bottom[1:-1, 2:] - board_thickness], dim=1)
    points_bottom = torch.cat([points_bottom, torch.stack([
            torch.tensor(-width/2 + spacing * (turns-1 + 1/2)),
            -heights[turns-1],
            torch.tensor(separation/2 - board_thickness/2)
        ]).unsqueeze(0)])

    points1 = torch.cat([points_top, torch.flip(points_bottom, dims=[0])])

    # Create mirrored coil (no in-place operations)
    points2 = torch.cat([points1[:, :2], points1[:, 2:] - separation], dim=1)
    
    # Create Coil objects
    gc1 = coilsim.Coil(points1)
    gc2 = coilsim.Coil(points2)

    return gc1, gc2

# raw_heights = torch.ones(turns, requires_grad=True)
gc1, gc2 = create_gradcoil(raw_heights,spacing, points_sc, points_combined)

roi = coilsim.ROI(0.005,0.05,-0.05, axis='y', point_density=800)

## Quiver Plots for Initial Guess

In [10]:
observation_points = roi.points
B = gc1.biot_savart(observation_points, 1) + gc2.biot_savart(observation_points, 1)

fig = plt.figure(figsize=(8, 6))
bounds = [[-0.1,0.1],[-0.1,0.1],[-0.1,0.1]]
ax = fig.add_subplot(111, projection='3d')

gc1.plot(show=False, ax=ax)
gc2.plot(show=False, ax=ax)
# roi.plot(show=False, ax=ax)
ax.set_xlim(bounds[0])
ax.set_ylim(bounds[1])
ax.set_zlim(bounds[2])
fac = 100
coilsim.plot_magnetic_field(observation_points, B.detach()[:,0]*fac, B.detach()[:,1]*fac, B.detach()[:,2]*fac, ax=ax, show=False)
# ax.set_box_aspect(aspect=(1, 1, 1))
plt.show()

## Optimization

In [11]:
raw_heights = torch.ones(turns, requires_grad=True)

optimizer = torch.optim.Adam([raw_heights], lr=0.1)

observation_points = roi.points

num_epochs = 1000
y_norm = observation_points[:,1]/torch.max(observation_points[:,1])
Bz = torch.zeros(len(observation_points[:,0]))

losses = []
slopes = []

In [12]:
with tqdm(range(num_epochs)) as titer:
    for i in titer:
        optimizer.zero_grad()
        gc1, gc2 = create_gradcoil(raw_heights,spacing,points_combined, points_sc)
        Bz = gc1.biot_savart(observation_points, 1)[:,2] + gc2.biot_savart(observation_points, 1)[:,2]
        # print(heights)
        bz_norm = Bz/torch.max(Bz)
        y = y_norm
        N = y.shape[0]
        slope = (N * torch.sum(y * Bz) - torch.sum(y) * torch.sum(Bz)) / (N * torch.sum(y**2) - torch.sum(y)**2)
        # print(Bz)
        slopes.append(slope.detach())
        loss = torch.sum((y_norm - bz_norm)**2)# - torch.sigmoid(slope*1000)
        # loss = torch.sum((y_norm + bz_norm)**2)
        # print(loss)z
        
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        titer.set_postfix(loss=loss.item())

print(f'Gradient Strength: {slopes[-1] * 20500}mT/m')


100%|██████████| 1000/1000 [00:10<00:00, 91.02it/s, loss=2.5]

Gradient Strength: 10.355120658874512mT/m


In [18]:
training_fig = plt.figure()
# plt.plot(np.log(losses))
plt.plot(np.array(slopes)*20.5195)
plt.xlabel("Training Step")
plt.ylabel("Log Loss")
plt.title("Training Processs")
plt.show()

## Plot Errors

In [19]:
observation_points = roi.points
heights_orig = torch.ones(turns, requires_grad=True)
gc1, gc2 = create_gradcoil(heights_orig,spacing,points_combined, points_sc)
B = gc1.biot_savart(observation_points, 1) + gc2.biot_savart(observation_points, 1)

gc3, gc4 = create_gradcoil(raw_heights,spacing,points_combined, points_sc)
Bn = gc3.biot_savart(observation_points, 1) + gc4.biot_savart(observation_points, 1)

all_data = np.concatenate([B.detach(), Bn.detach()])
vmin, vmax = np.min(all_data), np.max(all_data)
norm = Normalize(vmin=vmin, vmax=vmax)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(221, projection='3d')
gc1.plot(show=False, ax=ax)
gc2.plot(show=False, ax=ax)
ax.set_xlim(bounds[0])
ax.set_ylim(bounds[1])
ax.set_zlim(bounds[2])
ax.scatter(observation_points[:,0], observation_points[:,1], observation_points[:,2], c=B.detach()[:,2], norm=norm)

ax = fig.add_subplot(222, projection='3d')
bz_norm = B[:,2]/torch.max(Bn[:,2])
y_norm = observation_points[:,1]/torch.max(observation_points[:,1])

ax.scatter(observation_points[:,0], observation_points[:,1], observation_points[:,2], c=(bz_norm.detach()-y_norm) * torch.max(B.detach()[:,2]), norm=norm)
ax.set_xlim(bounds[0])
ax.set_ylim(bounds[1])
ax.set_zlim(bounds[2])



ax = fig.add_subplot(223, projection='3d')
gc3.plot(show=False, ax=ax)
gc4.plot(show=False, ax=ax)
ax.set_xlim(bounds[0])
ax.set_ylim(bounds[1])
ax.set_zlim(bounds[2])
ax.scatter(observation_points[:,0], observation_points[:,1], observation_points[:,2], c=Bn.detach()[:,2], norm=norm)

ax = fig.add_subplot(224, projection='3d')
bz_norm = Bn[:,2]/torch.max(Bn[:,2])
y_norm = observation_points[:,1]/torch.max(observation_points[:,1])
ax.scatter(observation_points[:,0], observation_points[:,1], observation_points[:,2], c=(bz_norm.detach()-y_norm) * torch.max(Bn.detach()[:,2]), norm=norm)
ax.set_xlim(bounds[0])
ax.set_ylim(bounds[1])
ax.set_zlim(bounds[2])
plt.tight_layout()
plt.show()

## Quiver Plot after Optimization 

In [20]:
observation_points = roi.points

fig = plt.figure(figsize=(8, 6))
bounds = [[-0.1,0.1],[-0.1,0.1],[-0.1,0.1]]
ax = fig.add_subplot(111, projection='3d')
gc3.plot(show=False, ax=ax)
roi.plot(show=False, ax=ax)
gc4.plot(show=False, ax=ax)
ax.set_xlim(bounds[0])
ax.set_ylim(bounds[1])
ax.set_zlim(bounds[2])
fac = 100
coilsim.plot_magnetic_field(observation_points, Bn.detach()[:,0]*fac, Bn.detach()[:,1]*fac, Bn.detach()[:,2]*fac, ax=ax, show=False)
# ax.set_box_aspect(aspect=(1, 1, 1))
plt.show()

In [21]:
#slice
sx = torch.tensor(0,dtype=torch.float32)
sy = torch.linspace(-length/2, length/2, 40,dtype=torch.float32)
sz = torch.arange(-0.005,0.005,(sy[1]-sy[0])/8,dtype=torch.float32)
X, Y, Z = torch.meshgrid(sx, sy, sz, indexing='ij')
slicepoints = torch.vstack((X.ravel(), Y.ravel(), Z.ravel())).T
Bs = gc3.biot_savart(slicepoints, 1) + gc4.biot_savart(slicepoints, 1)

Bo = gc1.biot_savart(slicepoints, 1) + gc2.biot_savart(slicepoints, 1)

slicefig = plt.figure()
plt.imshow(Bs.detach()[:,2].reshape(len(sy), -1).squeeze())

In [23]:
data = Bs.detach()[:,2].reshape(len(sy), -1).T.squeeze()
data_og = Bo.detach()[:,2].reshape(len(sy), -1).T.squeeze()
onedfig = plt.figure()
plt.plot(sy*1000, data[8,:]*1000)
plt.plot(sy*1000, data_og[8,:]*1000)
plt.xlabel("Y (mm)")
plt.ylabel("Field Strength (mT)")
plt.legend(["Optimized", "Original"])

In [98]:
import sexpdata
import yaml
import os

pcbfile = 'Rect_Grads_PCB/Rect_Grads_PCB.kicad_pcb'

def strip_coil(pcb):
    mdfile = pcb.split('.')[0] + '.yaml'
    
    boardlines = []
    with open(pcb, 'r') as board:
        boardlines = board.readlines()

    if (os.path.exists(mdfile)):
        with open(mdfile, 'r') as file:
            coilinfo = yaml.safe_load(file)
        with open(pcb, 'w') as board:
            for idx, line in enumerate(boardlines):
                if idx < coilinfo['start_line'] or idx > coilinfo['end_line']:
                    board.write(line)
        return coilinfo['start_line']
    else:
        return len(boardlines) -1

strip_coil(pcbfile)

307

In [74]:
def segment_string(start, end, width, layer, net):
    return f"""	(segment
		(start {start[0]} {start[1]})
		(end {end[0]} {end[1]})
		(width {width})
		(layer "{layer}")
		(net {net})
		(uuid "{str(uuid.uuid4())}")
	)\n"""

In [99]:
def write_coil(pcb, startline, coil, width, net, center):
    boardlines = []
    with open(pcb, 'r') as board:
        boardlines = board.readlines()

    with open(pcb, 'w') as board:
        for i in range(startline-1):
            board.write(boardlines[i])

        endline = startline
        for segment in coil.get_segments().detach().numpy():
            if(segment[0,2] != segment[1,2]): #Via, don't draw
                pass
            layer = "F.Cu" if segment[0,2] > np.mean(gc1.points.detach().numpy()[:,2]) else "B.Cu"
            board.write(segment_string(segment[0,0:2]*1000 + center, 
                                       segment[1,0:2]*1000 + center,
                                       width,layer,net))
            endline += 8

        for i in range(startline-1, len(boardlines)):
            board.write(boardlines[i])

    mdfile = pcb.split('.')[0] + '.yaml'
    with open(mdfile, 'w') as file:
        file.write(f'start_line: {startline}\n')
        file.write(f'end_line: {endline}')

write_coil(pcbfile, 307, gc3, 0.8, 1, np.array([100,100]))

In [85]:
np.mean(gc1.points.detach().numpy()[:,2])

0.015004907

In [ ]:
gc1.get_segments()[0][0,0:1]

tensor([[ 0.0105, -0.0894,  0.0158],
        [-0.0115, -0.0894,  0.0158]], grad_fn=<SelectBackward0>)

In [80]:
gc1.get_segments()

tensor([[[ 0.0105, -0.0894,  0.0158],
         [-0.0115, -0.0894,  0.0158]],

        [[-0.0115, -0.0894,  0.0158],
         [-0.0115, -0.0920,  0.0158]],

        [[-0.0115, -0.0920,  0.0158],
         [ 0.0115, -0.0920,  0.0158]],

        [[ 0.0115, -0.0920,  0.0158],
         [ 0.0115, -0.0805,  0.0158]],

        [[ 0.0115, -0.0805,  0.0158],
         [-0.0125, -0.0805,  0.0158]],

        [[-0.0125, -0.0805,  0.0158],
         [-0.0125, -0.0930,  0.0158]],

        [[-0.0125, -0.0930,  0.0158],
         [ 0.0125, -0.0930,  0.0158]],

        [[ 0.0125, -0.0930,  0.0158],
         [ 0.0125, -0.0715,  0.0158]],

        [[ 0.0125, -0.0715,  0.0158],
         [-0.0135, -0.0715,  0.0158]],

        [[-0.0135, -0.0715,  0.0158],
         [-0.0135, -0.0940,  0.0158]],

        [[-0.0135, -0.0940,  0.0158],
         [ 0.0135, -0.0940,  0.0158]],

        [[ 0.0135, -0.0940,  0.0158],
         [ 0.0135, -0.0626,  0.0158]],

        [[ 0.0135, -0.0626,  0.0158],
         [-0.0145, -0.0626

In [76]:
print(segment_string([0,1],[5,6],0.2, "F.Cu", 1))

	(segment
		(start 0 1)
		(end 5 6)
		(width 0.2)
		(layer "F.Cu")
		(net 1)
		(uuid "0206aad9-9708-421d-b78d-50a7cda3df68")
	)



In [49]:
mdfile = pcbfile.split('.')[0] + '.yaml'
with open(mdfile, 'r') as file:
    coilinfo = yaml.safe_load(file)

In [50]:
coilinfo

{'start_line': 349, 'end_line': 356}